# 配料：从重复的列，到基与维数

对应第七、八、九章。甲、乙、丙每份都是 100 克，蛋白质与脂肪分别为 (10,20)、(20,10)、(30,30) 克。这是假设成分，只研究指定的记录与约束。

只需要 NumPy 与 Matplotlib，从上到下运行。中文放在问题与输出中，图用英文避免缺少字体。每次先预测，再运行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "figure.dpi": 140})
a = np.array([10.0, 20.0])
b = np.array([20.0, 10.0])
c = np.array([30.0, 30.0])
A = np.column_stack([a, b, c])
BLUE, ORANGE, GREEN = "#2463a6", "#cb6b17", "#23845b"
assert np.allclose(c, a+b)


## 1. 新增丙，哪些量变了？

先比较甲 1 份、乙 2 份，与乙 1 份、丙 1 份。两项成分相同吗？总质量也相同吗？

再把下面 t 从 0 改成 0.5、1、1.5。每次预测三种用量、成分与质量。为什么最后一次等式还成立，配方却不能做？

In [ ]:
t = 0.5  # 再试 0、1、1.5
amounts = np.array([1-t, 2-t, t])
nutrients = A @ amounts
mass = 100*amounts.sum()
print("甲、乙、丙用量（份）:", amounts)
print("蛋白质、脂肪（克）:", nutrients)
print("总质量（克）:", mass)
print("用量都非负:", bool(np.all(amounts >= 0)))
assert np.allclose(nutrients, [50, 40])

parameters = np.linspace(-0.2, 1.5, 120)
family = np.vstack([1-parameters, 2-parameters, parameters])
outputs = A @ family
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for row, color, label in zip(family, [BLUE, ORANGE, GREEN], ["A", "B", "C"]):
    axes[0].plot(parameters, row, color=color, lw=2, label=label)
axes[0].axvspan(0, 1, color=GREEN, alpha=0.1)
axes[0].axhline(0, color="0.5", lw=0.8)
axes[0].set(xlabel="C portions: t", ylabel="Ingredient portions", title="Different recipes")
axes[0].legend()
for row, color, label in zip(outputs, [BLUE, ORANGE], ["Protein", "Fat"]):
    axes[1].plot(parameters, row, color=color, lw=2, label=label)
axes[1].set(xlabel="C portions: t", ylabel="Nutrient mass (g)", title="Same two outputs")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 2. 换一组基，系数怎么变？

保持目标不变。先用甲乙表示，再用乙丙表示。程序使用手算推导的公式，没有把求解器当成推导。

先试目标 (50,40)；再试 (10,20)，也就是一份甲。为什么两组基都能表示，只有其中一种非负配方可用？

In [ ]:
P, F = 50.0, 40.0  # 再试 (10,20)、(10,0)、(40,50)
target = np.array([P, F])
r, s = (2*F-P)/30, (2*P-F)/30
coeff_ab = np.array([r, s])
coeff_bc = np.array([s-r, r])
B_ab = np.column_stack([a, b])
B_bc = np.column_stack([b, c])
print("甲乙基坐标:", coeff_ab, "重建:", B_ab @ coeff_ab)
print("乙丙基坐标:", coeff_bc, "重建:", B_bc @ coeff_bc)
print("甲乙配方非负:", bool(np.all(coeff_ab >= -1e-10)))
print("乙丙配方非负:", bool(np.all(coeff_bc >= -1e-10)))
assert np.allclose(B_ab @ coeff_ab, target)
assert np.allclose(B_bc @ coeff_bc, target)


下面从非负用量取一些样本，两组基的采样范围为什么不同？图只显示有限样本，一般结论来自正文的坐标关系，不能只靠散点证明。

In [ ]:
portion = np.linspace(0, 3, 13)
x, y = np.meshgrid(portion, portion)
coefficients = np.vstack([x.ravel(), y.ravel()])
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
for ax, basis, title in zip(axes, [B_ab, B_bc], ["A/B nonnegative samples", "B/C nonnegative samples"]):
    samples = basis @ coefficients
    ax.scatter(samples[0], samples[1], s=12, alpha=0.45, color=BLUE)
    ax.scatter([target[0]], [target[1]], color=ORANGE, s=70, label="Chosen target", zorder=4)
    ax.set(xlim=(-5, 155), ylim=(-5, 155), aspect="equal",
           xlabel="Protein (g)", ylabel="Fat (g)", title=title)
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()


## 3. 删到只剩一列，会失去什么？

只留甲。尝试配出目标 (50,40)，先用蛋白质决定用量，再核对脂肪。

改成目标 (20,40) 后，为什么这次成功？只靠一列，是所有目标都不能达到，还是只能达到某一条线？

In [ ]:
target = np.array([50.0, 40.0])  # 再试 [20,40]
portion_from_protein = target[0]/a[0]
prediction = portion_from_protein*a
print("由蛋白质要求算出的甲用量:", portion_from_protein)
print("得到两项成分:", prediction)
print("目标:", target, "两项都满足:", np.allclose(prediction, target))


## 4. 记录总质量后，再检查相关关系

甲乙各一份，和丙一份的两项成分相同，但总质量不同。现在把质量作为第三项输出。

先手算原来的调整 (-1,-1,1) 对三个输出的影响，再运行。接着给定成分 (50,40)，分别要求总质量 300、250、200 克，算三种用量。再试 150 克，等式能成立吗，非负用量还能实现吗？

In [ ]:
A_mass = np.vstack([A, [100.0, 100.0, 100.0]])
old_adjustment = np.array([-1.0, -1.0, 1.0])
print("旧调整的三项输出变化（克）:", A_mass @ old_adjustment)
assert np.allclose(A_mass @ old_adjustment, [0, 0, -100])

P, F, M = 50.0, 40.0, 250.0  # 再试质量 300、200、150
t = (P+F)/30 - M/100
r = M/100 + (F-2*P)/30
s = M/100 + (P-2*F)/30
amounts = np.array([r, s, t])
target_three = np.array([P, F, M])
print("甲、乙、丙用量（份）:", amounts)
print("三项重建（克）:", A_mass @ amounts)
print("等式成立:", np.allclose(A_mass @ amounts, target_three))
print("可用非负配方:", bool(np.all(amounts >= -1e-10)))
assert np.allclose(A_mass @ amounts, target_three)

# 用数值秩作核对；线性无关的一般论证见正文，而不是靠此函数证明。
print("只记录两项成分时的独立方向数:", np.linalg.matrix_rank(A))
print("再记录总质量时的独立方向数:", np.linalg.matrix_rank(A_mass))
assert np.linalg.matrix_rank(A) == 2
assert np.linalg.matrix_rank(A_mass) == 3

# 多取几组目标，检查手推公式；负值也允许，因为此处测试实数关系。
for P, F, M in [(50,40,300), (50,40,200), (10,20,100), (-3,8,25), (0,0,0)]:
    formula = np.array([M/100+(F-2*P)/30, M/100+(P-2*F)/30, (P+F)/30-M/100])
    assert np.allclose(A_mass @ formula, [P,F,M])


## 合上实验，自己说明

1. 为什么三种原料的两项成分只有两个独立方向？给出非零的零输出调整。
2. 为什么甲乙、乙丙都是基？分别检查张成与线性无关。
3. 换基前后，什么没变、什么变了？实际非负配方范围一定不变吗？
4. 为什么加上总质量后，原来的重复关系消失？只检查这一条关系不再成立，是否足以证明三列无关？
5. 维数到底是原料数、记录字段数，还是描述指定空间所需的独立系数数目？

进一步：若丙每份不是 100 克，而是 200 克，三项记录会怎样改变？修改质量行，重新检查原来的调整。